# 실험

In [3]:
import requests

import os
from dotenv import load_dotenv

load_dotenv()

# 아래 API 입력
REST_API_KEY = os.getenv("KAKAO_API_KEY")

url = 'https://apis-navi.kakaomobility.com/v1/directions'
headers = {
    'Authorization': f'KakaoAK {REST_API_KEY}'
}
params = {
    'origin': '127.1086228,37.4012191',       # 출발지
    'destination': '127.1086403,37.4021397',  # 도착지
    'priority': 'RECOMMEND',
}

response = requests.get(url, headers=headers, params=params)
data = response.json()

# 경로 정보 추출
for section in data['routes'][0]['sections']:
    for road in section['roads']:
        print(f"{road['name']} / 거리: {road['distance']}m / 시간: {road['duration'] // 1000}s")

 / 거리: 31m / 시간: 0s
판교역로 / 거리: 114m / 시간: 0s
판교역로 / 거리: 167m / 시간: 0s
판교역로241번길 / 거리: 92m / 시간: 0s
 / 거리: 15m / 시간: 0s


# 기록기록
- `NotAuthorizedError` : 카카오맵 권한은 앱 하나에만 신청 가능한 듯 (비즈니스 권한 없이)


# 경로 

## 주소 -> 위경도

In [4]:
import requests
import folium

# 1.주소 -> 위경도 변환 함수
def geocode(address):
    """주소 → 위경도 변환 (카카오 로컬 API)"""
    url = 'https://dapi.kakao.com/v2/local/search/address.json'
    headers = {'Authorization': f'KakaoAK {REST_API_KEY}'}
    params = {'query': address}
    
    response = requests.get(url, headers=headers, params=params)
    result = response.json()
    # print(result)
    
    if result['documents']:
        x = result['documents'][0]['x']  # 경도
        y = result['documents'][0]['y']  # 위도
        print("위경도 좌표 추출 성공!")
        return float(x), float(y)
    else:
        raise ValueError("주소를 찾을 수 없습니다ㅜㅜ")

# 잘 되는지 실험 
add = "서울특별시 강남구 역삼동 826-21"

geocode(add)


위경도 좌표 추출 성공!


(127.029292206472, 37.4955529336109)

## 경로찾기

In [5]:

# 2. 경로찾기 함수 - road_details false
def get_directions(origin, destination, waypoints=None):
    """
        자동차 길찾기 API
        waypoints : 경유지, 리스트로 5개까지 넣을 수 있음!
    """

    url = 'https://apis-navi.kakaomobility.com/v1/directions'
    headers = {'Authorization': f'KakaoAK {REST_API_KEY}'}
    params = {
        'origin': f'{origin[0]},{origin[1]}',
        'destination': f'{destination[0]},{destination[1]}',
        'priority': "RECOMMEND",
        # 'road_details' : 'true',
    }

    if waypoints:
        # 경유지를 "x,y" 형식의 문자열 리스트로 변환 
        waypoints_str = [f"{wp[0]},{wp[1]}" for wp in waypoints]
        params['waypoints'] = waypoints_str
        
    response = requests.get(url, headers=headers, params=params)
    result = response.json()
    # print(result)
    return result

# 실험 
origin_address = "서울특별시 강남구 역삼동 826-21"
destination_address = "서울특별시 강남구 삼성동 159"
origin_coord = geocode(origin_address)
destination_coord = geocode(destination_address)

get_directions(origin_coord, destination_coord)

위경도 좌표 추출 성공!
위경도 좌표 추출 성공!


{'trans_id': '0197170bda5076db8fbd555473c7fe63',
 'routes': [{'result_code': 0,
   'result_msg': '길찾기 성공',
   'summary': {'origin': {'name': '',
     'x': 127.02928413452449,
     'y': 37.495549035548684},
    'destination': {'name': '',
     'x': 127.05881355730037,
     'y': 37.51252060777642},
    'waypoints': [],
    'priority': 'RECOMMEND',
    'bound': {'min_x': 127.02795122733251,
     'min_y': 37.49542994874317,
     'max_x': 127.06082819784154,
     'max_y': 37.51423062760874},
    'fare': {'taxi': 10900, 'toll': 0},
    'distance': 4784,
    'duration': 1034},
   'sections': [{'distance': 4784,
     'duration': 1034,
     'bound': {'min_x': 127.05990908031762,
      'min_y': 37.495438039450185,
      'max_x': 127.06084671337271,
      'max_y': 37.51422477953356},
     'roads': [{'name': '강남대로',
       'distance': 250,
       'duration': 47,
       'traffic_speed': 21.0,
       'traffic_state': 3,
       'vertexes': [127.02893499568054,
        37.495438039450185,
        127.

## 시각화 함수

In [6]:

def visualize_route(origin, destination, route_data, waypoints=None):
    """Folium 지도 위에 경로 시각화"""
    # 지도 생성 (출발지 중심)
    m = folium.Map(location=[origin[1], origin[0]], zoom_start=14)

    # 출발지, 도착지 마커
    folium.Marker([origin[1], origin[0]], tooltip="출발지", icon=folium.Icon(color='green')).add_to(m)
    folium.Marker([destination[1], destination[0]], tooltip="도착지", icon=folium.Icon(color='red')).add_to(m)

    # 경유지 마커 
    if waypoints:
        for idx, wp in enumerate(waypoints):
            folium.Marker([wp[1], wp[0]], tooltip=f"경유지 {idx+1}", icon=folium.Icon(color='blue')).add_to(m)
    
    # 경로 폴리라인 좌표 추출
    sections = route_data['routes'][0]['sections']
    for section in sections:
        for road in section['roads']:
            coords = road['vertexes']  # [x1, y1, x2, y2, ...]
            points = [(coords[i+1], coords[i]) for i in range(0, len(coords), 2)]
            folium.PolyLine(points, color='blue', weight=5).add_to(m)

    return m

# 시각화

In [ ]:
# 단일경로 시각화

# 주소 입력
origin_address = "서울특별시 마포구 모래내로 3길 3"
destination_address = "경상남도 통영시 멘데해안길 205" # 통영 이순신공원

# 3. 주소 → 위경도
origin_coord = geocode(origin_address)
destination_coord = geocode(destination_address)

# 4. 길찾기 API 요청
route_data = get_directions(origin_coord, destination_coord)

# 5. 지도에 경로 시각화
map_view = visualize_route(origin_coord, destination_coord, route_data)

# 6. HTML 파일로 저장 or 노트북에서 바로 보기
# map_view.save("kakao_route_map.html")
map_view  # Jupyter에서는 자동 렌더링

위경도 좌표 추출 성공!
위경도 좌표 추출 성공!


# N km 별로 좌표 뽑기 및 시각화

In [11]:
from math import radians, cos, sin, asin, sqrt
import folium

def haversine(coord1, coord2):
    """두 위경도 좌표 사이 거리 (단위: km)"""
    lon1, lat1, lon2, lat2 = map(radians, [coord1[0], coord1[1], coord2[0], coord2[1]])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return 6371 * c

def extract_5km_points(route_data, interval_km=5):
    """경로 데이터에서 5km마다 좌표 추출"""
    coords = []
    for section in route_data['routes'][0]['sections']:
        for road in section['roads']:
            vertexes = road['vertexes']
            for i in range(0, len(vertexes) - 2, 2):
                x1, y1 = vertexes[i], vertexes[i+1]
                x2, y2 = vertexes[i+2], vertexes[i+3]
                coords.append(((x1, y1), (x2, y2)))

    full_path = [coords[0][0]]
    for segment in coords:
        full_path.append(segment[1])

    extracted = []
    accumulated = 0
    next_target = interval_km

    for i in range(1, len(full_path)):
        d = haversine(full_path[i-1], full_path[i])
        accumulated += d

        if accumulated >= next_target:
            extracted.append(full_path[i])
            next_target += interval_km

    return extracted

def visualize_route_with_5km(origin, destination, route_data, waypoints=None):
    """Folium 지도 위에 경로 + 5km마다 마커 시각화"""
    m = folium.Map(location=[origin[1], origin[0]], zoom_start=14)

    # 출발지, 도착지 마커
    folium.Marker([origin[1], origin[0]], tooltip="출발지", icon=folium.Icon(color='green')).add_to(m)
    folium.Marker([destination[1], destination[0]], tooltip="도착지", icon=folium.Icon(color='red')).add_to(m)

    # 경유지 마커 
    if waypoints:
        for idx, wp in enumerate(waypoints):
            folium.Marker([wp[1], wp[0]], tooltip=f"경유지 {idx+1}", icon=folium.Icon(color='blue')).add_to(m)
    
    # 경로 폴리라인
    sections = route_data['routes'][0]['sections']
    for section in sections:
        for road in section['roads']:
            coords = road['vertexes']
            points = [(coords[i+1], coords[i]) for i in range(0, len(coords), 2)]
            folium.PolyLine(points, color='blue', weight=5).add_to(m)

    # 5km마다 마커
    interval_points = extract_5km_points(route_data, interval_km=5)
    for i, pt in enumerate(interval_points):
        folium.CircleMarker(
            location=[pt[1], pt[0]],
            radius=3,  # 점 크기
            color='purple',
            fill=True,
            fill_color='purple',
            fill_opacity=1,
            tooltip=f"{(i+1)*5}km 지점"
        ).add_to(m)
    return m

In [8]:
# 주소 입력
origin_address = "서울특별시 종로구 효자로 12"
destination_address = "경남 통영시 충렬로 33"  # 통영 이순신공원

# 위경도 변환
origin_coord = geocode(origin_address)
destination_coord = geocode(destination_address)

# 경로 데이터 요청
route_data = get_directions(origin_coord, destination_coord)

# 시각화 (5km마다 마커 포함)
map_view = visualize_route_with_5km(origin_coord, destination_coord, route_data)

# HTML로 저장 또는 Jupyter에서 보기
# map_view.save("route_with_5km_markers.html")
map_view

위경도 좌표 추출 성공!
위경도 좌표 추출 성공!


In [ ]:
interval_points = extract_5km_points(route_data, interval_km=5)
print(interval_points)
# points = []
# for i, point in enumerate(interval_points):
#     points.append(point)
#     print(f"{(i+1)*5}km 지점: {point}")

# print(points)

[(126.99999918905259, 37.544330897558446), (127.01816252061731, 37.503701771404366), (127.03916340731878, 37.46412878936889), (127.08323872172886, 37.41686653952456), (127.10218291010248, 37.38855599473638), (127.10326507302938, 37.34603323713007), (127.10348832047815, 37.30160236014302), (127.10376463116029, 37.25357617946718), (127.09591345005207, 37.213444601945305), (127.09147320922787, 37.16653511530046), (127.10729497264947, 37.11944601386889), (127.12613270303927, 37.08413625579521), (127.13681331133334, 37.04484421718685), (127.15318788027434, 36.99666138308551), (127.18089508481894, 36.956610748341035), (127.18886351584455, 36.923097631296756), (127.18758152253208, 36.873579387175766), (127.17435035208358, 36.83458507307928), (127.16996922806496, 36.79180320019847), (127.20584728999698, 36.765688796041175), (127.25979541905103, 36.732593779914495), (127.29723950925994, 36.73091925426243), (127.34809555207215, 36.714554651921105), (127.37370325491938, 36.67632819574277), (127.3

# 실험 (무시)

## 경유지 추가

In [41]:
# 경유지 추가 

# 주소 입력
origin_address = "서울특별시 강남구 역삼동 826-21"
destination_address = "서울특별시 강남구 삼성동 159"
waypoint_addresses = ["서울특별시 서초구 서초동 1321-1", "서울특별시 송파구 잠실동 40-1"]

# 주소 → 위경도 변환
origin_coord = geocode(origin_address)
destination_coord = geocode(destination_address)
waypoints_coord = [geocode(addr) for addr in waypoint_addresses]

# 길찾기 API 요청
route_data = get_directions(origin_coord, destination_coord, waypoints=waypoints_coord)

# 지도에 경로 시각화
map_view = visualize_route(origin_coord, destination_coord, route_data, waypoints=waypoints_coord)
map_view 

{'documents': [{'address': {'address_name': '서울 강남구 역삼동 826-21', 'b_code': '1168010100', 'h_code': '1168064000', 'main_address_no': '826', 'mountain_yn': 'N', 'region_1depth_name': '서울', 'region_2depth_name': '강남구', 'region_3depth_h_name': '역삼1동', 'region_3depth_name': '역삼동', 'sub_address_no': '21', 'x': '127.029292206472', 'y': '37.4955529336109'}, 'address_name': '서울 강남구 역삼동 826-21', 'address_type': 'REGION_ADDR', 'road_address': {'address_name': '서울 강남구 강남대로 364', 'building_name': '미왕빌딩', 'main_building_no': '364', 'region_1depth_name': '서울', 'region_2depth_name': '강남구', 'region_3depth_name': '역삼동', 'road_name': '강남대로', 'sub_building_no': '', 'underground_yn': 'N', 'x': '127.029293901519', 'y': '37.4955498697675', 'zone_no': '06241'}, 'x': '127.029292206472', 'y': '37.4955529336109'}], 'meta': {'is_end': True, 'pageable_count': 1, 'total_count': 1}}
위경도 좌표 추출 성공!
{'documents': [{'address': {'address_name': '서울 강남구 삼성동 159', 'b_code': '1168010500', 'h_code': '1168058000', 'main_addre

In [ ]:
# section 내 정보 확인용 실험 

import folium
import random

def visualize_route(origin, destination, route_data):
    """Folium 지도 위에 section별 다른 색상으로 경로 시각화"""
    m = folium.Map(location=[origin[1], origin[0]], zoom_start=14)

    # 출발지와 도착지 마커
    folium.Marker([origin[1], origin[0]], tooltip="출발지", icon=folium.Icon(color='green')).add_to(m)
    folium.Marker([destination[1], destination[0]], tooltip="도착지", icon=folium.Icon(color='red')).add_to(m)

    # 색상 팔레트
    colors = [
        'blue', 'orange', 'purple', 'darkred', 'green', 'black',
        'cadetblue', 'darkgreen', 'pink', 'gray'
    ]

    roads = route_data['routes'][0]['sections'][0]['roads']

    for idx, road in enumerate(roads):
        coords = road['vertexes']
        points = [(coords[i + 1], coords[i]) for i in range(0, len(coords), 2)]

        color = colors[idx % len(colors)]  # 색상 순환
        folium.PolyLine(points, color=color, weight=5, tooltip=road['name']).add_to(m)

    return m

In [21]:
map_view = visualize_route(origin_coord, destination_coord, route_data)
map_view